## Jupyter Notebook — dARK + Authority Test Script

### Flow:
1. Connect to the network and load deployed contracts
2. Create 2 new wallets (authority-1 and authority-2)
3. Fund each wallet with 10 ETH from the master wallet
4. Register the wallets as Authorities in the smart contract
5. Authorize NAANs and create ARKs
6. Resolve and verify

In [4]:
import json
import time
import configparser
from pathlib import Path
from web3 import Web3, HTTPProvider
from eth_account import Account
from web3.middleware import geth_poa_middleware

In [5]:
# ==========================================
# NETWORK CONFIGURATION
# ==========================================

# Read RPC URL from config.ini
config = configparser.ConfigParser()
config.read("../../config.ini")
blockchain_net = config['base']['blockchain_net']
bc_config      = config[blockchain_net]
RPC_URL        = bc_config['url']

# ---- Master wallet (pre-funded in the Besu genesis block) ----
# Acts as both the contract admin AND the ETH funding source
MASTER_ADDRESS = "0x28eC26EbE1d460c4c9541Fb7595a0031fFf1BEc1"
MASTER_KEY     = "0xc8bd0f575e9f64c8a509772b18035c40ad75aab145ce9c588a83ddbcc8f96fcc"

# The master wallet IS the contract admin (set at deploy time via constructor)
ADMIN_ADDRESS = MASTER_ADDRESS
ADMIN_KEY     = MASTER_KEY

# Connect to the Besu node
web3 = Web3(HTTPProvider(RPC_URL))
web3.middleware_onion.inject(geth_poa_middleware, layer=0)
web3.eth.default_account = web3.to_checksum_address(ADMIN_ADDRESS)

print(f"Network       : {blockchain_net}")
print(f"RPC URL       : {RPC_URL}")
print(f"Connected     : {web3.is_connected()}")
print(f"Chain ID      : {web3.eth.chain_id}")
print(f"Master wallet : {MASTER_ADDRESS}")
master_balance = web3.from_wei(web3.eth.get_balance(MASTER_ADDRESS), 'ether')
print(f"Master balance: {master_balance} ETH")

Network       : dark-local
RPC URL       : http://localhost:8545
Connected     : True
Chain ID      : 2025
Master wallet : 0x28eC26EbE1d460c4c9541Fb7595a0031fFf1BEc1
Master balance: 999999.84967479976995 ETH


In [6]:
# ==========================================
# LOAD DEPLOYED CONTRACTS
# ==========================================

deployed = configparser.ConfigParser()
deployed.read("../deployed_contracts.ini")

authority_address = web3.to_checksum_address(deployed["Authority"]["address"])
authority_abi     = json.loads(deployed["Authority"]["abi"])
dark_address      = web3.to_checksum_address(deployed["dARK"]["address"])
dark_abi          = json.loads(deployed["dARK"]["abi"])

authority = web3.eth.contract(address=authority_address, abi=authority_abi)
dark      = web3.eth.contract(address=dark_address, abi=dark_abi)

# Verify that the on-chain admin matches the master wallet
on_chain_admin = authority.functions.admin().call()
assert on_chain_admin.lower() == ADMIN_ADDRESS.lower(), (
    f"Admin mismatch! On-chain: {on_chain_admin} | expected: {ADMIN_ADDRESS}"
)

print(f"Authority contract : {authority_address}")
print(f"dARK contract      : {dark_address}")
print(f"✅ On-chain admin matches master wallet: {on_chain_admin}")

Authority contract : 0xB4F5786036d993E898a621e9096e8a9A328238Da
dARK contract      : 0xB619eC3E49F7E39006732438D3B9dC83b29280e8
✅ On-chain admin matches master wallet: 0x28eC26EbE1d460c4c9541Fb7595a0031fFf1BEc1


In [7]:
# ==========================================
# HELPERS: TRANSACTIONS AND REVERT DECODING
# ==========================================

def get_revert_reason(tx_hash):
    """Decode the revert reason string from a failed transaction."""
    try:
        tx     = web3.eth.get_transaction(tx_hash)
        replay = web3.eth.call(
            {"from": tx["from"], "to": tx["to"], "data": tx["input"], "value": tx["value"]},
            tx["blockNumber"] - 1
        )
        return replay
    except Exception as e:
        # The exception message from eth_call contains the revert string
        return str(e)

def send_transaction(contract_func, sender, private_key, gas=200_000):
    """Build, sign, and send a smart contract transaction. Decodes revert reason on failure."""
    txn = contract_func.build_transaction({
        "from"    : sender,
        "nonce"   : web3.eth.get_transaction_count(sender),
        "gas"     : gas,
        "gasPrice": web3.to_wei("40", "gwei"),
        "chainId" : web3.eth.chain_id,
    })
    signed  = web3.eth.account.sign_transaction(txn, private_key)
    # Compatible with eth-account < 0.12 (rawTransaction) and >= 0.12 (raw_transaction)
    raw_tx  = getattr(signed, 'raw_transaction', None) or signed.rawTransaction
    tx_hash = web3.eth.send_raw_transaction(raw_tx)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    if receipt["status"] == 0:
        reason = get_revert_reason(tx_hash)
        raise Exception(f"Transaction reverted: {reason}")
    return receipt, tx_hash

def transfer_eth(to_address, amount_ether, sender=MASTER_ADDRESS, sender_key=MASTER_KEY):
    """Transfer ETH from the master wallet to a target address."""
    tx = {
        "from"    : sender,
        "to"      : web3.to_checksum_address(to_address),
        "value"   : web3.to_wei(amount_ether, "ether"),
        "gas"     : 21000,
        "gasPrice": web3.to_wei("40", "gwei"),
        "nonce"   : web3.eth.get_transaction_count(sender),
        "chainId" : web3.eth.chain_id,
    }
    signed  = web3.eth.account.sign_transaction(tx, sender_key)
    raw_tx  = getattr(signed, 'raw_transaction', None) or signed.rawTransaction
    tx_hash = web3.eth.send_raw_transaction(raw_tx)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    return receipt, tx_hash

print("✅ Helpers loaded")

✅ Helpers loaded


In [8]:
# ==========================================
# CREATE AUTHORITY WALLETS
# Generates two fresh key pairs for authority-1 and authority-2.
# UUIDs include a timestamp suffix to avoid re-registration conflicts
# when the notebook is re-run against the same blockchain state.
# ==========================================

ts = int(time.time())  # unique suffix for this run

auth1_account = Account.create()
auth2_account = Account.create()

AUTH1_ADDRESS = auth1_account.address
AUTH1_KEY     = auth1_account.key.hex()
uuid1         = f"authority-001-{ts}"
naan1         = "11111"

AUTH2_ADDRESS = auth2_account.address
AUTH2_KEY     = auth2_account.key.hex()
uuid2         = f"authority-002-{ts}"
naan2         = "22222"

print("=== Authority 1 ===")
print(f"  UUID    : {uuid1}")
print(f"  Address : {AUTH1_ADDRESS}")
print(f"  Key     : {AUTH1_KEY}")
print(f"  NAAN    : {naan1}")
print()
print("=== Authority 2 ===")
print(f"  UUID    : {uuid2}")
print(f"  Address : {AUTH2_ADDRESS}")
print(f"  Key     : {AUTH2_KEY}")
print(f"  NAAN    : {naan2}")

=== Authority 1 ===
  UUID    : authority-001-1778231902
  Address : 0x0bDC3d852F09baB768Ccc91C7ae699544686107b
  Key     : 0xe30785e6695defbc7d23ebd788378a3a8c0e9ff62e2ea5d9fe411bde71dbe4dd
  NAAN    : 11111

=== Authority 2 ===
  UUID    : authority-002-1778231902
  Address : 0xe09C9b97047169EaA4a090f4F1B26aD66Cee2168
  Key     : 0x7b3eb78c71122bfbaec7ae97c64c81a8e58f5c6db53719278a6342b9dca764d5
  NAAN    : 22222


In [9]:
# ==========================================
# FUND AUTHORITY WALLETS WITH 10 ETH EACH
# ETH comes from the master wallet
# ==========================================

# Fund Authority 1
receipt, tx_hash = transfer_eth(AUTH1_ADDRESS, 10)
balance1 = web3.from_wei(web3.eth.get_balance(AUTH1_ADDRESS), 'ether')
print(f"✅ Authority 1 funded | tx: {tx_hash.hex()} | balance: {balance1} ETH")

# Fund Authority 2
receipt, tx_hash = transfer_eth(AUTH2_ADDRESS, 10)
balance2 = web3.from_wei(web3.eth.get_balance(AUTH2_ADDRESS), 'ether')
print(f"✅ Authority 2 funded | tx: {tx_hash.hex()} | balance: {balance2} ETH")

✅ Authority 1 funded | tx: 0x146ce89cd2eb26f8981bf0f57e5db55756f14e2b869f84ea53781578028304f8 | balance: 10 ETH
✅ Authority 2 funded | tx: 0xb00c564706d148a0be0ade1a9963e1d8bedea2d1dfd7feabbcb69c2ab2144352 | balance: 10 ETH


In [10]:
# ==========================================
# REGISTER AUTHORITIES IN THE CONTRACT
# Only the contract admin (master wallet) can call register_authority.
# The encrypted_private_key field is stored on-chain for key recovery;
# in production it must be a real AES-256 encrypted key.
# ==========================================

# Register Authority 1
receipt, tx_hash = send_transaction(
    authority.functions.register_authority(uuid1, AUTH1_ADDRESS, f"enc-key-{AUTH1_KEY}"),
    ADMIN_ADDRESS,
    ADMIN_KEY
)
print(f"✅ Authority 1 registered | uuid={uuid1} | tx: {tx_hash.hex()}")

# Register Authority 2
receipt, tx_hash = send_transaction(
    authority.functions.register_authority(uuid2, AUTH2_ADDRESS, f"enc-key-{AUTH2_KEY}"),
    ADMIN_ADDRESS,
    ADMIN_KEY
)
print(f"✅ Authority 2 registered | uuid={uuid2} | tx: {tx_hash.hex()}")

✅ Authority 1 registered | uuid=authority-001-1778231902 | tx: 0x07132ba883ccf173df8072c5897f56346e5911851afb8d941946b457f7793571
✅ Authority 2 registered | uuid=authority-002-1778231902 | tx: 0xa8b7244e6f36e6afd44d35327c7ca5d129da9a422153869118795054687210c7


In [11]:
# ==========================================
# AUTHORIZE NAANs
# Each authority claims its own NAAN by signing from its own wallet.
# ==========================================

receipt, tx_hash = send_transaction(
    authority.functions.authorize_naan(naan1),
    AUTH1_ADDRESS,
    AUTH1_KEY
)
print(f"✅ Authority 1 authorized NAAN '{naan1}' | tx: {tx_hash.hex()}")

receipt, tx_hash = send_transaction(
    authority.functions.authorize_naan(naan2),
    AUTH2_ADDRESS,
    AUTH2_KEY
)
print(f"✅ Authority 2 authorized NAAN '{naan2}' | tx: {tx_hash.hex()}")

✅ Authority 1 authorized NAAN '11111' | tx: 0x2e555a34284de49b0b12292556d3c4bf556b8a3ea03915a675d409d1a6f23064
✅ Authority 2 authorized NAAN '22222' | tx: 0xda8962d0d8847906acfe4111f9a9e5d69d86789d84f8aab694e7a638447a2870


In [12]:
# ==========================================
# CREATE ARKs
# Each authority mints an identifier under its authorized NAAN.
# ARK format: ark:/NAAN/name
# ==========================================

# ARK for Authority 1
ark1_name = f"doc-auth1-{ts}"
ark1_url  = "https://institution-auth1.org/doc-001"
ark1_cid  = "QmCidAuth1Example"

receipt, tx_hash = send_transaction(
    dark.functions.create_ark(naan1, ark1_name, ark1_url, ark1_cid),
    AUTH1_ADDRESS,
    AUTH1_KEY,
    gas=500_000
)
print(f"✅ ARK created: ark:/{naan1}/{ark1_name} | tx: {tx_hash.hex()}")

# ARK for Authority 2
ark2_name = f"doc-auth2-{ts}"
ark2_url  = "https://institution-auth2.org/doc-001"
ark2_cid  = "QmCidAuth2Example"

receipt, tx_hash = send_transaction(
    dark.functions.create_ark(naan2, ark2_name, ark2_url, ark2_cid),
    AUTH2_ADDRESS,
    AUTH2_KEY,
    gas=500_000
)
print(f"✅ ARK created: ark:/{naan2}/{ark2_name} | tx: {tx_hash.hex()}")

✅ ARK created: ark:/11111/doc-auth1-1778231902 | tx: 0x7753f1d747adcffca0089039d66cd41fa8a596200585ed5baba60ad0ec7a9ba1
✅ ARK created: ark:/22222/doc-auth2-1778231902 | tx: 0xf281e8b758a1cf692a57ff44582531ce353d6a42611a5174da6463ec3a1a3c08


In [13]:
# ==========================================
# RESOLVE AND VERIFY ARKs
# ==========================================

print("=== Resolution ===")
url1 = dark.functions.resolve(naan1, ark1_name).call()
print(f"  ark:/{naan1}/{ark1_name} → {url1}")

url2 = dark.functions.resolve(naan2, ark2_name).call()
print(f"  ark:/{naan2}/{ark2_name} → {url2}")

print()
print("=== Full data — ARK 1 ===")
ark_data = dark.functions.get_ark(naan1, ark1_name).call()
print(f"  Name    : {ark_data[0]}")
print(f"  NAAN    : {ark_data[1]}")
print(f"  URL     : {ark_data[2]}")
print(f"  CID     : {ark_data[3]}")
print(f"  Owner   : {ark_data[4]}")
print(f"  Created : {ark_data[5]}")
print(f"  Updated : {ark_data[6]}")

print()
print("=== Full data — ARK 2 ===")
ark_data2 = dark.functions.get_ark(naan2, ark2_name).call()
print(f"  Name    : {ark_data2[0]}")
print(f"  NAAN    : {ark_data2[1]}")
print(f"  URL     : {ark_data2[2]}")
print(f"  CID     : {ark_data2[3]}")
print(f"  Owner   : {ark_data2[4]}")
print(f"  Created : {ark_data2[5]}")
print(f"  Updated : {ark_data2[6]}")

=== Resolution ===
  ark:/11111/doc-auth1-1778231902 → https://institution-auth1.org/doc-001
  ark:/22222/doc-auth2-1778231902 → https://institution-auth2.org/doc-001

=== Full data — ARK 1 ===
  Name    : doc-auth1-1778231902
  NAAN    : 11111
  URL     : https://institution-auth1.org/doc-001
  CID     : QmCidAuth1Example
  Owner   : 0x0bDC3d852F09baB768Ccc91C7ae699544686107b
  Created : 1778231916
  Updated : 1778231916

=== Full data — ARK 2 ===
  Name    : doc-auth2-1778231902
  NAAN    : 22222
  URL     : https://institution-auth2.org/doc-001
  CID     : QmCidAuth2Example
  Owner   : 0xe09C9b97047169EaA4a090f4F1B26aD66Cee2168
  Created : 1778231918
  Updated : 1778231918


In [14]:
# ==========================================
# FINAL SUMMARY
# ==========================================

print("╔══════════════════════════════════════════════════════╗")
print("║          dARK 2.0 — Test Complete ✅                 ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print(f"Master / Admin : {MASTER_ADDRESS}")
print()
print(f"Authority 1    : {AUTH1_ADDRESS}")
print(f"  UUID         : {uuid1}")
print(f"  NAAN         : {naan1}")
print(f"  ARK          : ark:/{naan1}/{ark1_name}")
print()
print(f"Authority 2    : {AUTH2_ADDRESS}")
print(f"  UUID         : {uuid2}")
print(f"  NAAN         : {naan2}")
print(f"  ARK          : ark:/{naan2}/{ark2_name}")

# Final balances
b1 = web3.from_wei(web3.eth.get_balance(AUTH1_ADDRESS), 'ether')
b2 = web3.from_wei(web3.eth.get_balance(AUTH2_ADDRESS), 'ether')
bm = web3.from_wei(web3.eth.get_balance(MASTER_ADDRESS), 'ether')
print()
print(f"Master balance : {bm} ETH")
print(f"Auth 1 balance : {b1} ETH")
print(f"Auth 2 balance : {b2} ETH")

╔══════════════════════════════════════════════════════╗
║          dARK 2.0 — Test Complete ✅                 ║
╚══════════════════════════════════════════════════════╝

Master / Admin : 0x28eC26EbE1d460c4c9541Fb7595a0031fFf1BEc1

Authority 1    : 0x0bDC3d852F09baB768Ccc91C7ae699544686107b
  UUID         : authority-001-1778231902
  NAAN         : 11111
  ARK          : ark:/11111/doc-auth1-1778231902

Authority 2    : 0xe09C9b97047169EaA4a090f4F1B26aD66Cee2168
  UUID         : authority-002-1778231902
  NAAN         : 22222
  ARK          : ark:/22222/doc-auth2-1778231902

Master balance : 999979.83309951976995 ETH
Auth 1 balance : 9.9860742 ETH
Auth 2 balance : 9.9860742 ETH
